# 📊 Task 2: Unemployment Analysis with Python
**Dataset:** unemployment.csv — Indian state-wise unemployment data (Jan–Nov 2020) including Covid-19 period.

Columns: Region, Date, Frequency, Estimated Unemployment Rate (%), Estimated Employed, Estimated Labour Participation Rate (%), longitude, latitude

In [ ]:
# !pip install pandas matplotlib seaborn plotly

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 110
print("Libraries loaded!")

## 2. Load the Real Dataset

In [ ]:
# Load directly from GitHub
url = "https://raw.githubusercontent.com/amankharwal/Website-data/master/unemployment.csv"
df = pd.read_csv(url)

# OR local: df = pd.read_csv("unemployment.csv")

# Clean column names (some have leading spaces)
df.columns = df.columns.str.strip()
print("Shape:", df.shape)
print("Columns:", list(df.columns))
df.head()

In [ ]:
print("Missing values:")
print(df.isnull().sum())
print()
df.dtypes

## 3. Data Cleaning & Preprocessing

In [ ]:
# Rename for convenience
df.rename(columns={
    'Estimated Unemployment Rate (%)':      'Unemployment_Rate',
    'Estimated Employed':                   'Employed',
    'Estimated Labour Participation Rate (%)': 'Labour_Participation'
}, inplace=True)

# Parse date
df['Date'] = pd.to_datetime(df['Date'].str.strip(), dayfirst=True)
df['Month'] = df['Date'].dt.month
df['Month_Name'] = df['Date'].dt.strftime('%b')

# Remove duplicate Region column if present
cols = list(df.columns)
if cols.count('Region') > 1:
    df = df.loc[:, ~df.columns.duplicated()]

print("Cleaned shape:", df.shape)
print("Date range:", df['Date'].min().date(), "→", df['Date'].max().date())
df.head()

## 4. Exploratory Data Analysis

In [ ]:
print("Unemployment Rate Stats:")
print(df['Unemployment_Rate'].describe())
print()
print("Avg Unemployment Rate by Region:")
print(df.groupby('Region')['Unemployment_Rate'].mean().sort_values(ascending=False).round(2))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Distribution
axes[0,0].hist(df['Unemployment_Rate'], bins=25, color='#E74C3C', edgecolor='white')
axes[0,0].set_title('Unemployment Rate Distribution', fontweight='bold')
axes[0,0].set_xlabel('Rate (%)')

# 2. By Region (box)
regions = df['Region'].unique()
colors  = ['#3498DB','#E74C3C','#2ECC71','#9B59B6','#E67E22']
for i, (region, color) in enumerate(zip(regions, colors)):
    subset = df[df['Region']==region]['Unemployment_Rate']
    axes[0,1].boxplot(subset, positions=[i], patch_artist=True,
                      boxprops=dict(facecolor=color, alpha=0.7))
axes[0,1].set_xticks(range(len(regions))); axes[0,1].set_xticklabels(regions, rotation=15)
axes[0,1].set_title('Unemployment Rate by Region', fontweight='bold')
axes[0,1].set_ylabel('Rate (%)')

# 3. Top 10 states
state_avg = df.groupby('Region')['Unemployment_Rate'].mean().sort_values(ascending=False).head(10)
state_avg.plot(kind='barh', ax=axes[1,0], color='#E74C3C', alpha=0.8)
axes[1,0].set_title('Top 10 Highest Avg Unemployment Regions', fontweight='bold')
axes[1,0].set_xlabel('Avg Rate (%)')

# 4. Monthly trend
monthly = df.groupby('Month')['Unemployment_Rate'].mean()
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
axes[1,1].bar([month_names[m-1] for m in monthly.index], monthly.values, color='#3498DB', alpha=0.8)
axes[1,1].set_title('Avg Unemployment by Month (2020)', fontweight='bold')
axes[1,1].set_ylabel('Rate (%)')
axes[1,1].axhline(monthly.mean(), color='red', linestyle='--', label=f'Mean: {monthly.mean():.1f}%')
axes[1,1].legend()

plt.tight_layout()
plt.savefig('unemployment_eda.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Covid-19 Impact Analysis

In [ ]:
# Pre-Covid: Jan-Mar 2020, During: Apr-Jun 2020, Post: Jul-Nov 2020
df['Covid_Phase'] = pd.cut(df['Month'],
    bins=[0, 3, 6, 12],
    labels=['Pre-Covid (Jan-Mar)', 'Covid Peak (Apr-Jun)', 'Recovery (Jul-Nov)'])

phase_stats = df.groupby('Covid_Phase')['Unemployment_Rate'].agg(['mean','max','min']).round(2)
print("Covid Impact Summary:")
print(phase_stats)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors_p = ['#2ECC71','#E74C3C','#3498DB']
for i, (phase, color) in enumerate(zip(df['Covid_Phase'].cat.categories, colors_p)):
    data = df[df['Covid_Phase']==phase]['Unemployment_Rate']
    axes[0].hist(data, alpha=0.6, bins=15, label=phase, color=color)
axes[0].set_title('Unemployment Distribution by Covid Phase', fontweight='bold')
axes[0].set_xlabel('Rate (%)'); axes[0].legend(fontsize=9)

phase_means = df.groupby('Covid_Phase')['Unemployment_Rate'].mean()
bars = axes[1].bar(range(len(phase_means)), phase_means.values, color=colors_p, alpha=0.8)
axes[1].set_xticks(range(len(phase_means)))
axes[1].set_xticklabels(phase_means.index, rotation=10, ha='right', fontsize=9)
axes[1].set_title('Average Unemployment Rate by Phase', fontweight='bold')
axes[1].set_ylabel('Avg Rate (%)')
for bar, v in zip(bars, phase_means.values):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2, f'{v:.1f}%', ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('covid_impact.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. State-wise Heatmap

In [ ]:
pivot = df.pivot_table(values='Unemployment_Rate', index='Region', columns='Month', aggfunc='mean').round(1)
month_labels = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov']
pivot.columns = month_labels[:len(pivot.columns)]

plt.figure(figsize=(13, 7))
sns.heatmap(pivot, cmap='RdYlGn_r', annot=True, fmt='.1f', linewidths=0.3,
            cbar_kws={'label': 'Unemployment Rate (%)'})
plt.title('Region × Month Unemployment Heatmap (India 2020)', fontsize=13, fontweight='bold')
plt.ylabel('Region'); plt.xlabel('Month')
plt.tight_layout()
plt.savefig('unemployment_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Correlation Analysis

In [ ]:
numeric_cols = ['Unemployment_Rate', 'Employed', 'Labour_Participation', 'longitude', 'latitude']
corr = df[numeric_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', square=True, linewidths=0.5)
plt.title('Correlation Heatmap', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('unemployment_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Scatter: Unemployment vs Labour Participation

In [ ]:
plt.figure(figsize=(10, 6))
colors_r = ['#3498DB','#E74C3C','#2ECC71','#9B59B6','#E67E22']
for region, color in zip(df['Region'].unique(), colors_r):
    sub = df[df['Region']==region]
    plt.scatter(sub['Unemployment_Rate'], sub['Labour_Participation'],
                label=region, color=color, alpha=0.7, s=50)
plt.xlabel('Unemployment Rate (%)')
plt.ylabel('Labour Participation Rate (%)')
plt.title('Unemployment vs Labour Participation by Region', fontsize=13, fontweight='bold')
plt.legend(loc='best', fontsize=9)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('unemployment_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Key Policy Insights

In [ ]:
print("=" * 55)
print("KEY POLICY INSIGHTS — Indian Unemployment 2020")
print("=" * 55)
print(f"Overall avg unemployment rate : {df['Unemployment_Rate'].mean():.2f}%")
print(f"Peak month (Covid)            : {df.groupby('Month')['Unemployment_Rate'].mean().idxmax()} — {df.groupby('Month')['Unemployment_Rate'].mean().max():.1f}%")
print(f"Best month                    : {df.groupby('Month')['Unemployment_Rate'].mean().idxmin()} — {df.groupby('Month')['Unemployment_Rate'].mean().min():.1f}%")
print(f"Highest region avg            : {df.groupby('Region')['Unemployment_Rate'].mean().idxmax()}")
print()
print("INSIGHTS:")
print("  • Covid-19 caused a massive spike in Apr-May 2020")
print("  • Urban regions showed sharper spikes than rural")
print("  • Labour participation also dropped during Covid peak")
print("  • Recovery began post-June 2020 but remained above pre-Covid levels")